# 6장 1강: CV 기반 하이퍼파라미터 튜닝과 모델 탐색

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 하이퍼파라미터 탐색과 교차검증을 결합해 최적 모델을 찾습니다.

- GridSearchCV와 RandomizedSearchCV의 탐색 방식과 비용 차이를 비교합니다.
- Optuna의 TPE 탐색을 이용해 초기 탐색 이후 이전 trial 결과를 활용하는 탐색을 수행합니다.
- 교차검증 평균 점수와 표준편차를 함께 확인합니다.
- 탐색 시간까지 고려하여 최종 후보를 선정합니다.
- 처음부터 분리해 둔 hold-out test set은 최종 모델을 정한 뒤 한 번만 사용합니다.
- 최종 모델과 메타데이터를 저장합니다.

> 이 실습에서는 6장 1강 교안에서 다룬 Grid Search, Random Search, Optuna, CV 평균·표준편차, 탐색 비용, hold-out test, 모델 저장 내용만 사용합니다.


## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- scipy
- Optuna
- joblib
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

하이퍼파라미터 탐색에 Test 데이터를 사용하면 안 됩니다.

따라서 먼저 전체 데이터를 **Train / hold-out Test**로 분리하고,

- 하이퍼파라미터 탐색과 교차검증 → Train 데이터만 사용
- 최종 성능 확인 → 모든 선택이 끝난 뒤 hold-out Test를 한 번만 사용

하는 흐름으로 진행합니다.

In [1]:
import os
import json
import time
import joblib
import optuna
import numpy as np     
import pandas as pd

from datetime import datetime
from scipy.stats import randint

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Hold-out Test 크기:", X_test.shape)

c:\exe_6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train 크기: (1168, 12)
Hold-out Test 크기: (292, 12)


---

# 필수 1. Grid Search와 Random Search 비교

## 배경

GridSearchCV는 지정한 모든 조합을 빠짐없이 평가하고, RandomizedSearchCV는 정해진 횟수만큼 조합을 무작위로 선택합니다.

이번 실습에서는 같은 RandomForestRegressor를 대상으로 두 탐색 전략을 비교합니다.

## 문제 1. GridSearchCV를 수행하세요.

### 요구사항

1. `RandomForestRegressor(random_state=42, n_jobs=-1)`를 사용합니다.
2. 다음 후보를 사용합니다.

```python
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}
```

3. `cv=5`를 사용합니다.
4. `scoring="r2"`를 사용합니다.
5. 탐색 시간을 측정합니다.
6. `best_params_`와 `best_score_`를 출력합니다.
7. `cv_results_`에서 최적 조합의 `mean_test_score`, `std_test_score`를 확인합니다.

### 확인 포인트

- 총 9개 조합을 모두 평가하는가?
- 5-Fold이므로 각 조합이 5번 평가되는 구조를 이해했는가?
- 평균 점수뿐 아니라 표준편차도 확인했는가?

In [2]:
# TODO
# GridSearchCV를 수행하세요.
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1)

grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]}

grid = GridSearchCV(rf,
    grid_params, cv=5,
    scoring="r2",
    n_jobs=-1)

start = time.time()
grid.fit(X_train, y_train)
grid_seconds = time.time() - start
grid_best_idx = grid.best_index_
grid_cv_mean = grid.cv_results_[
    "mean_test_score"
][grid_best_idx]

grid_cv_std = grid.cv_results_[
    "std_test_score"
][grid_best_idx]

print("최적 파라미터:", grid.best_params_)
print("최고 CV 평균:", grid_cv_mean)
print("CV 표준편차:", grid_cv_std)
print("탐색 시간:", grid_seconds)

최적 파라미터: {'max_depth': 12, 'n_estimators': 200}
최고 CV 평균: 0.8206179847869421
CV 표준편차: 0.06442538938888975
탐색 시간: 3.966564178466797


In [3]:
grid_result_df = pd.DataFrame(grid.cv_results_)

print("평가한 조합 수:", len(grid.cv_results_["params"]))
print("총 학습 횟수  :", len(grid.cv_results_["params"]) * 5)

print("\n[Grid 전체 후보 결과]")
display(
    grid_result_df[
        ["params", "mean_test_score", "std_test_score", "rank_test_score"]
    ].sort_values("rank_test_score")
)

평가한 조합 수: 9
총 학습 횟수  : 45

[Grid 전체 후보 결과]


,params,mean_test_score,std_test_score,rank_test_score
7,"{'max_depth': 12, 'n_estimators': 200}",0.820618,0.064425,1
6,"{'max_depth': 12, 'n_estimators': 100}",0.820348,0.065524,2
8,"{'max_depth': 12, 'n_estimators': 300}",0.820182,0.065787,3
5,"{'max_depth': 8, 'n_estimators': 300}",0.818850,0.063274,4
4,"{'max_depth': 8, 'n_estimators': 200}",0.817849,0.064449,5
3,"{'max_depth': 8, 'n_estimators': 100}",0.817761,0.065969,6
2,"{'max_depth': 4, 'n_estimators': 300}",0.780580,0.050523,7
1,"{'max_depth': 4, 'n_estimators': 200}",0.779586,0.049544,8
0,"{'max_depth': 4, 'n_estimators': 100}",0.779369,0.051308,9


#### 최고 CV 평균
#### 0.82, max_depth(12)와 n_estimators(200)의 조합으로 5개의 검증 R²를 평균한 값

#### CV 표준편차
#### 0.06, 최적 조합 Fold별로 검증 점수가 평균 주변에서 얼마나 달랐는지 나타냄

### Q1. Grid Search의 가장 큰 장점과 한계는 무엇인가요?
**답변:**  
#### 장점 : 지정한 조합을 빠짐없이 평가, 하이퍼 파라미터 튜닝 최적화를 용이하게 찾음
#### 단점 : 후보 수가 많아지면 탐색비용이 매우 느려짐

## 문제 2. RandomizedSearchCV를 수행하세요.

### 요구사항

1. 같은 RandomForestRegressor를 사용합니다.
2. 다음 탐색 공간을 사용합니다.

```python
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}
```

3. `n_iter=9`를 사용합니다.
4. `cv=5`, `scoring="r2"`를 사용합니다.
5. `random_state=42`를 사용합니다.
6. 탐색 시간을 측정합니다.
7. 최적 파라미터, CV 평균 점수, 표준편차를 확인합니다.

### 확인 포인트

- 전체 조합을 모두 보는 것이 아니라 9번만 무작위로 탐색하는가?
- `n_iter`가 탐색 예산을 직접 정한다는 점을 이해했는가?

In [4]:
# TODO
# RandomizedSearchCV를 수행하세요.
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}

rand = RandomizedSearchCV(
    rf,
    random_params,
    n_iter=9,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

start = time.time()
rand.fit(X_train, y_train)
rand_seconds = time.time() - start

rand_best_idx = rand.best_index_
rand_cv_mean = rand.cv_results_["mean_test_score"][rand_best_idx]
rand_cv_std = rand.cv_results_["std_test_score"][rand_best_idx]

print("최적 파라미터:", rand.best_params_)
print("CV 평균      :", rand_cv_mean)
print("CV 표준편차  :", rand_cv_std)
print("탐색 시간    :", rand_seconds)

rand_result_df = pd.DataFrame(rand.cv_results_)

print("\n[Random 전체 후보 결과]")
display(
    rand_result_df[
        ["params", "mean_test_score", "std_test_score", "rank_test_score"]
    ].sort_values("rank_test_score")
)

최적 파라미터: {'max_depth': 11, 'n_estimators': 230}
CV 평균      : 0.8210296383331108
CV 표준편차  : 0.06508647306696069
탐색 시간    : 1.6860196590423584

[Random 전체 후보 결과]


,params,mean_test_score,std_test_score,rank_test_score
6,"{'max_depth': 11, 'n_estimators': 230}",0.821030,0.065086,1
1,"{'max_depth': 11, 'n_estimators': 288}",0.820919,0.066358,2
4,"{'max_depth': 11, 'n_estimators': 216}",0.819858,0.065923,3
0,"{'max_depth': 10, 'n_estimators': 279}",0.818594,0.069279,4
2,"{'max_depth': 8, 'n_estimators': 202}",0.818300,0.063981,5
7,"{'max_depth': 9, 'n_estimators': 152}",0.817948,0.067975,6
5,"{'max_depth': 7, 'n_estimators': 203}",0.816651,0.062739,7
3,"{'max_depth': 6, 'n_estimators': 174}",0.809847,0.061638,8
8,"{'max_depth': 5, 'n_estimators': 187}",0.801043,0.055673,9


#### Grid 서치의 CV 평균보다 약 0.0004 높다.
#### 반면에 표준편차는 0.006 정도로 커졌다.
#### Grid 서치와 지금 모델을 비교하여 보면 평균 R2은 높지만 표준편차(변동)가 조금 더 크다.

### Q2. Random Search가 Grid Search보다 유리할 수 있는 상황은 언제인가요?
**답변:**  
#### 내가 어떤 값을 넣어야 할 지 모르고, 모든 조합을 평가할 시간이 없을때 주로 유용하다. 

---

# 필수 2. Optuna와 CV 결합

## 배경

Grid와 Random은 이전 탐색 결과와 상관없이 후보를 선택합니다.

이번 실습에서는 Optuna의 `TPESampler`를 사용합니다.  
처음 3개의 trial은 초기 탐색으로 수행하고, 이후 trial부터는 앞선 결과를 참고하여 다음 후보를 제안하도록 설정합니다.

## 문제 1. Optuna objective에 교차검증을 연결하세요.

### 요구사항

1. `objective(trial)` 함수를 만듭니다.
2. 다음 하이퍼파라미터를 탐색합니다.
   - `n_estimators`: 100 ~ 300
   - `max_depth`: 4 ~ 12
3. `RandomForestRegressor`를 사용합니다.
4. `cross_val_score(..., cv=5, scoring="r2")`를 사용합니다.
5. CV 평균 점수를 반환합니다.
6. 다음 설정의 `TPESampler`를 만듭니다.

```python
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)
```

7. `direction="maximize"`와 위 sampler를 사용해 study를 만듭니다.
8. `n_trials=9`로 탐색합니다.
9. 탐색 시간을 측정합니다.
10. `best_params`, `best_value`를 출력합니다.

### 확인 포인트

- 한 번의 시도를 trial이라고 부르는가?
- 탐색 전체를 study라고 부르는가?
- 처음 3개 trial 이후에는 이전 trial 결과를 활용하는 TPE 탐색이 이루어지는가?
- CV 평균 성능을 Optuna의 최적화 대상으로 사용했는가?


In [5]:
# TODO
# Optuna objective + CV를 구성하세요.
# TPESampler(seed=42, n_startup_trials=3)를 사용하고
# 총 9 trial을 실행하세요.
def objective(trial):
    params = {"n_estimators": trial.suggest_int(
            "n_estimators",
            100,300),
        "max_depth": trial.suggest_int(
            "max_depth", 4, 12)}

    model = RandomForestRegressor(
        **params, random_state=42, n_jobs=-1)

    scores = cross_val_score(
        model, X_train, y_train,
        cv=5,
        scoring="r2",
        n_jobs=-1)

    return scores.mean()


sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler)

start = time.time()
study.optimize( objective,
    n_trials=9)

optuna_seconds = time.time() - start

print("최적 파라미터:", study.best_params)
print("최고 CV 평균:", study.best_value)
print("탐색 시간:", optuna_seconds)

[I 2026-10-12 09:51:03,755] A new study created in memory with name: no-name-486a679d-3c3e-4850-8490-b8806f954bf2
[I 2026-10-12 09:51:03,964] Trial 0 finished with value: 0.8212542415716282 and parameters: {'n_estimators': 175, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-12 09:51:04,223] Trial 1 finished with value: 0.8184595590761907 and parameters: {'n_estimators': 247, 'max_depth': 9}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-12 09:51:04,370] Trial 2 finished with value: 0.7990289774524457 and parameters: {'n_estimators': 131, 'max_depth': 5}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-12 09:51:04,528] Trial 3 finished with value: 0.8201477777785264 and parameters: {'n_estimators': 109, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-12 09:51:04,770] Trial 4 finished with value: 0.8205950100254196 and parameters: {'n_estimators': 204, 'max_depth': 12}. Best is trial 0 with value: 0.821254

최적 파라미터: {'n_estimators': 175, 'max_depth': 12}
최고 CV 평균: 0.8212542415716282
탐색 시간: 1.9331626892089844


### Q3. Optuna가 Grid / Random과 다른 핵심은 무엇인가요?

**답변:**  
#### Grid는 정해 둔 모든 후보 조합을 평가하고, Random은 정해진 횟수만큼 후보를 무작위로 선택
#### Optuna는 초기 3개 trial 이후 결과를 참고하여 상대적으로 유리한 영역을 더 깊게 탐색하는 방식을 사용

## 문제 2. Optuna 최적 조합의 CV 표준편차를 확인하세요.

### 요구사항

1. `study.best_params`로 RandomForestRegressor를 만듭니다.
2. 다시 5-Fold `cross_val_score()`를 수행합니다.
3. CV 평균과 표준편차를 계산합니다.

### 확인 포인트

- 최종 후보를 볼 때 평균만 보지 않고 Fold 간 흔들림도 확인했는가?

In [6]:
# TODO
# Optuna 최적 조합의 CV 평균과 표준편차를 확인하세요.
from sklearn.model_selection import KFold, cross_validate
optuna_best_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1)

optuna_scores = cross_val_score(
    optuna_best_model,
    X_train, y_train,
    cv=5,
    scoring="r2",
    n_jobs=-1)

optuna_cv_mean = optuna_scores.mean()
optuna_cv_std = optuna_scores.std()

print("Fold별 R2  :", np.round(optuna_scores, 4))
print("Optuna CV 평균:", optuna_cv_mean)
print("Optuna CV 표준편차:", optuna_cv_std)

Fold별 R2  : [0.8694 0.6967 0.828  0.8428 0.8694]
Optuna CV 평균: 0.8212542415716282
Optuna CV 표준편차: 0.06428073023916543



#### 앞서 진행한 세 가지 모델 결과 비교
#### Grid의 표준편차는 약 0.064425, Random은 0.06508였고 Optuna는 0.0642이다.
#### Optuna를 통해 결과를 구했을 때 평균이 가장 높고, 표준편차가 가장 적었다.

---

# 과제 1. 최종 모델 선정과 hold-out Test 확인

## 배경

세 탐색 전략에서 각각 최적 후보를 얻었습니다.

최종 모델은 **CV 평균을 우선 기준으로 비교**하되, 성능 차이가 매우 작다면 다음 항목도 함께 확인합니다.

- CV 표준편차
- 탐색 시간

그리고 최종 후보를 정한 뒤 **hold-out Test는 딱 한 번만** 사용합니다.

> 탐색 시간은 각 탐색 도구의 내부 동작과 실행 환경에 따라 포함되는 작업이 조금 다를 수 있으므로, 이번 실습에서는 절대적인 속도 순위가 아니라 **대략적인 탐색 비용 비교**로 해석합니다.

## 요구사항

1. Grid / Random / Optuna 결과를 다음 컬럼으로 하나의 DataFrame에 정리합니다.
   - `search_method`
   - `best_params`
   - `cv_mean`
   - `cv_std`
   - `search_seconds`
2. 세 후보를 비교합니다.
3. CV 평균을 우선 기준으로 최종 후보를 선택하고, 평균이 거의 비슷하다면 CV 표준편차와 탐색 비용도 함께 고려하여 선택 이유를 설명합니다.
4. 튜닝하지 않은 기준 모델(`RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)`)과 3번에서 선택한 튜닝 후보를 비교하세요. 필수 실습의 `cv=5`와 동일한 `KFold(n_splits=5, shuffle=False)` 분할을 두 모델에 공통 적용하고, `cross_validate(..., scoring="r2", return_train_score=True)`로 평가하세요.
   - Fold별 Train R²와 검증 R²를 `fold_comparison`으로 정리합니다.
   - Train R² 평균, 검증 R² 평균·표준편차, `Train 평균 - 검증 평균`을 `tuning_comparison`으로 정리합니다.
   - `튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²`를 계산합니다. R² 차이값으로 보고하며, 백분율 개선율로 바꾸지 않습니다.
   - 평균·변동·Train–검증 차이로 개선 여부를 설명합니다. 튜닝이 개선되지 않았다면 기준 모델 유지도 가능합니다. Test 확인 전에 최종 모델과 선택 이유를 확정합니다.
   - 최종 선택 모델을 Train 전체에 다시 학습합니다.
5. hold-out Test에서 `.score()`로 최종 R²를 딱 한 번 확인합니다.
6. 최종 선택 모델의 CV 평균과 hold-out Test R²의 차이를 계산하고, 과적합 가능성과 일반화 판단의 한계를 설명합니다. Test 결과를 본 뒤 다시 튜닝하거나 후보를 교체하지 않습니다.
7. 선택한 모델을 `model_artifacts/best_model.pkl`로 저장합니다.
8. 다음 정보를 `model_artifacts/model_meta.json`에 저장합니다.
   - 모델 이름
   - 저장 시각
   - 선택한 탐색 방법
   - 최적 파라미터
   - CV 평균
   - CV 표준편차
   - 최종 Test R²
   - 기준 모델 설정, 튜닝 후보 설정·탐색 방법
   - 튜닝 전후 CV 비교 결과와 CV 평균 R² 차이
   - 선택 모델 CV 평균과 Test R²의 차이

### 확인 포인트

- Test 데이터를 탐색 중간에 사용하지 않았는가?
- CV 평균을 우선으로 보고, 필요할 때 표준편차와 탐색 시간도 함께 고려했는가?
- 최종 모델을 정한 뒤 Test를 한 번만 확인했는가?
- 모델뿐 아니라 메타데이터도 저장했는가?

- 튜닝 전후를 동일한 Fold에서 비교하고 개선 여부를 수치로 설명했는가?
- CV의 `test_score`는 Fold 내부 검증 점수이며 hold-out Test 점수가 아님을 구분했는가?
- 작은 표준편차나 CV/Test 점수의 일치만으로 일반화가 보장된다고 단정하지 않았는가?


In [7]:
# TODO
# 세 탐색 후보를 비교하고 기준 모델과 튜닝 후보의 동일 Fold CV 결과를 정리하세요.
# 튜닝 전후 R² 차이를 계산해 최종 선택을 확정한 뒤
# 선택 모델만 hold-out Test에서 한 번 평가하고 모델·비교 메타데이터를 저장하세요.
search_summary = pd.DataFrame([
    {
        "search_method": "GridSearchCV",
        "best_params": grid.best_params_,
        "cv_mean": grid_cv_mean,
        "cv_std": grid_cv_std,
        "search_seconds": grid_seconds
    },
    {
        "search_method": "RandomizedSearchCV",
        "best_params": rand.best_params_,
        "cv_mean": rand_cv_mean,
        "cv_std": rand_cv_std,
        "search_seconds": rand_seconds
    },
    {
        "search_method": "Optuna",
        "best_params": study.best_params,
        "cv_mean": optuna_cv_mean,
        "cv_std": optuna_cv_std,
        "search_seconds": optuna_seconds
    }
])

print("[세 탐색 전략 비교]")
display(search_summary)

# CV 평균이 1순위, 평균이 같으면 표준편차가 작은 쪽을 선택합니다.
best_row = search_summary.sort_values(
    ["cv_mean", "cv_std"],
    ascending=[False, True]
).iloc[0]

selected_method = best_row["search_method"]
selected_params = best_row["best_params"]

print("선택한 탐색 방법:", selected_method)
print("선택한 파라미터  :", selected_params)
print("CV 평균          :", best_row["cv_mean"])
print("CV 표준편차      :", best_row["cv_std"])
print("탐색 시간        :", best_row["search_seconds"])

print("\nCV 평균 최대-최소 차이:", search_summary["cv_mean"].max() - search_summary["cv_mean"].min())
print("CV 표준편차 평균      :", search_summary["cv_std"].mean())


[세 탐색 전략 비교]


,search_method,best_params,cv_mean,cv_std,search_seconds
0,GridSearchCV,"{'max_depth': 12, 'n_estimators': 200}",0.820618,0.064425,3.966564
1,RandomizedSearchCV,"{'max_depth': 11, 'n_estimators': 230}",0.821030,0.065086,1.686020
2,Optuna,"{'n_estimators': 175, 'max_depth': 12}",0.821254,0.064281,1.933163


선택한 탐색 방법: Optuna
선택한 파라미터  : {'n_estimators': 175, 'max_depth': 12}
CV 평균          : 0.8212542415716282
CV 표준편차      : 0.06428073023916543
탐색 시간        : 1.9331626892089844

CV 평균 최대-최소 차이: 0.0006362567846860401
CV 표준편차 평균      : 0.06459753089833863


In [8]:
# 필수 실습의 cv=5와 동일한 분할을 두 모델에 공통 적용합니다.
from sklearn.model_selection import KFold, cross_validate
cv_common = KFold(n_splits=5, shuffle=False)

base_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

tuned_model = RandomForestRegressor(
    **selected_params,
    random_state=42,
    n_jobs=-1
)

base_cv = cross_validate(
    base_model, X_train, y_train,
    cv=cv_common, scoring="r2", return_train_score=True
)

tuned_cv = cross_validate(
    tuned_model, X_train, y_train,
    cv=cv_common, scoring="r2", return_train_score=True
)

fold_comparison = pd.DataFrame({
    "fold": range(1, 6),
    "base_train_r2": base_cv["train_score"],
    "base_valid_r2": base_cv["test_score"],
    "tuned_train_r2": tuned_cv["train_score"],
    "tuned_valid_r2": tuned_cv["test_score"]
})

print("[Fold별 Train / 검증 R2 비교]")
display(fold_comparison)

tuning_comparison = pd.DataFrame([
    {
        "model": "기준 모델",
        "train_r2_mean": base_cv["train_score"].mean(),
        "valid_r2_mean": base_cv["test_score"].mean(),
        "valid_r2_std": base_cv["test_score"].std(),
        "train_minus_valid": base_cv["train_score"].mean() - base_cv["test_score"].mean()
    },
    {
        "model": "튜닝 후보",
        "train_r2_mean": tuned_cv["train_score"].mean(),
        "valid_r2_mean": tuned_cv["test_score"].mean(),
        "valid_r2_std": tuned_cv["test_score"].std(),
        "train_minus_valid": tuned_cv["train_score"].mean() - tuned_cv["test_score"].mean()
    }
])

print("[튜닝 전후 비교]")
display(tuning_comparison)

# R2 차이값으로 보고합니다. 백분율로 바꾸지 않습니다.
cv_gain = tuned_cv["test_score"].mean() - base_cv["test_score"].mean()

print("튜닝 후 CV 평균 R2 - 기준 모델 CV 평균 R2 =", cv_gain)

# 개선되었으면 튜닝 후보, 아니면 기준 모델을 유지합니다.
if cv_gain > 0:
    final_model = tuned_model
    final_choice = "튜닝 후보"
    final_params = selected_params
    final_cv_mean = tuned_cv["test_score"].mean()
    final_cv_std = tuned_cv["test_score"].std()
else:
    final_model = base_model
    final_choice = "기준 모델"
    final_params = {"n_estimators": 100, "max_depth": None}
    final_cv_mean = base_cv["test_score"].mean()
    final_cv_std = base_cv["test_score"].std()

print("최종 선택    :", final_choice)
print("최종 파라미터:", final_params)

# 최종 선택 모델을 Train 전체에 다시 학습합니다.
final_model.fit(X_train, y_train)

print("최종 모델을 Train 전체에 다시 학습 완료")

[Fold별 Train / 검증 R2 비교]


,fold,base_train_r2,base_valid_r2,tuned_train_r2,tuned_valid_r2
0,1,0.971728,0.862667,0.971451,0.869414
1,2,0.975444,0.680790,0.976382,0.696696
2,3,0.974969,0.834270,0.972136,0.827994
3,4,0.974586,0.845045,0.972621,0.842770
4,5,0.971663,0.872282,0.970872,0.869396


[튜닝 전후 비교]


,model,train_r2_mean,valid_r2_mean,valid_r2_std,train_minus_valid
0,기준 모델,0.973678,0.819011,0.070370,0.154667
1,튜닝 후보,0.972692,0.821254,0.064281,0.151438


튜닝 후 CV 평균 R2 - 기준 모델 CV 평균 R2 = 0.0022435439729445728
최종 선택    : 튜닝 후보
최종 파라미터: {'n_estimators': 175, 'max_depth': 12}
최종 모델을 Train 전체에 다시 학습 완료


In [9]:
# 모든 선택이 끝난 뒤, hold-out Test를 딱 한 번만 사용합니다.
test_r2 = final_model.score(X_test, y_test)

gap = final_cv_mean - test_r2

print("[hold-out Test 최종 확인]")
print("CV 평균 R2    :", final_cv_mean)
print("Test R2       :", test_r2)
print("CV - Test 차이:", gap)

[hold-out Test 최종 확인]
CV 평균 R2    : 0.8212542415716282
Test R2       : 0.8910932915341476
CV - Test 차이: -0.06983904996251944


In [10]:
os.makedirs("model_artifacts", exist_ok=True)

joblib.dump(final_model, "model_artifacts/best_model.pkl")

model_meta = {
    "model_name": "ames_rf_tuned",
    "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "search_method": selected_method,
    "best_params": {k: (None if v is None else int(v)) for k, v in final_params.items()},
    "cv_mean": float(final_cv_mean),
    "cv_std": float(final_cv_std),
    "test_r2": float(test_r2),
    "base_model_config": {
        "n_estimators": 100,
        "max_depth": None,
        "random_state": 42,
        "n_jobs": -1
    },
    "tuned_candidate_config": {k: (None if v is None else int(v)) for k, v in selected_params.items()},
    "tuning_comparison": {
        "base_valid_r2_mean": float(base_cv["test_score"].mean()),
        "tuned_valid_r2_mean": float(tuned_cv["test_score"].mean()),
        "cv_mean_r2_gain": float(cv_gain)
    },
    "cv_minus_test": float(gap),
    "final_choice": final_choice
}

with open("model_artifacts/model_meta.json", "w", encoding="utf-8") as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=2)

print("[저장 완료]")
print("model_artifacts/best_model.pkl")
print("model_artifacts/model_meta.json")

[저장 완료]
model_artifacts/best_model.pkl
model_artifacts/model_meta.json


In [11]:
loaded_model = joblib.load("model_artifacts/best_model.pkl")

with open("model_artifacts/model_meta.json", "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)

print("불러온 모델 Test R2:", loaded_model.score(X_test, y_test))
display(loaded_meta)

불러온 모델 Test R2: 0.8910932915341476


{'model_name': 'ames_rf_tuned',
 'saved_at': '2026-10-12 09:51:23',
 'search_method': 'Optuna',
 'best_params': {'n_estimators': 175, 'max_depth': 12},
 'cv_mean': 0.8212542415716282,
 'cv_std': 0.06428073023916543,
 'test_r2': 0.8910932915341476,
 'base_model_config': {'n_estimators': 100,
  'max_depth': None,
  'random_state': 42,
  'n_jobs': -1},
 'tuned_candidate_config': {'n_estimators': 175, 'max_depth': 12},
 'tuning_comparison': {'base_valid_r2_mean': 0.8190106975986836,
  'tuned_valid_r2_mean': 0.8212542415716282,
  'cv_mean_r2_gain': 0.0022435439729445728},
 'cv_minus_test': -0.06983904996251944,
 'final_choice': '튜닝 후보'}

### Q4. CV 평균이 아주 조금 높은 모델의 탐색 시간이 몇 배 더 길다면 무조건 그 모델을 선택해야 하나요?

**답변:**  
아니요. CV 평균 차이는 약 0.0006인데, Fold 간 표준편차는 약 0.064로 약 100배 차이가 난다. 데이터를 다르게 나누기만 해도 순위가 뒤집힐 수 있는 크기라서, 이 차이를 진짜 성능 차이로 보기 어렵다.
CV 평균만 보고 고르는 게 아니라, 그 차이가 믿을 만한 크기인지 먼저 확인하고 비용을 함께 봐야 한다.

### Q5. hold-out Test를 탐색 마지막에 한 번만 사용해야 하는 이유는 무엇인가요?

**답변:**  
한 번이라도 탐색에 참여한 데이터는 더 이상 "처음 보는 데이터"가 아니기 때문이다. Test 점수로 파라미터를 바꾸면, 결국 Test에 잘 맞는 조합이 되어 점수가 실제보다 부풀려진다.
그래서 탐색과 교차검증은 Train 데이터로만 하고, 모든 선택이 끝난 뒤 hold-out Test로 최종 성능을 한 번만 확인한다.

### Q6. 튜닝 전후 CV 평균 R²는 얼마나 달라졌나요? Fold별 변동과 Train–검증 점수 차이를 근거로 개선 여부를 설명하세요.

**답변:**
#### CV 평균 R² 차이
기준 모델 0.819011 → 튜닝 후보 0.821254로, 약 0.0022 상승
#### Fold별 변동
Fold 1, 2는 튜닝 후보가 높지만 Fold 3, 4, 5는 기준 모델이 높다. 
#### Train–검증 차이
`train_minus_valid`가 기준 모델 약 0.1547, 튜닝 후보 약 0.1514이다. Train R²는 0.97대인데 검증은 0.82대로 나온다. `max_depth=12`일 경우 Train R²은 0.0033정도 줄었다.


### Q7. CV 결과와 최종 Test 결과를 어떻게 해석했나요? 과적합 가능성과 일반화 판단의 한계를 설명하세요.
**답변:**
CV 평균 R²는 약 0.8213, hold-out Test R²는 약 0.8910으로 Test가 약 0.0698 더 높게 나왔다. CV의 Fold 2가 약 0.6967로 다른 회차에 비해 낮은 점수가 나와 CV 평균 점수에 반영되어있다. 
Train R²는 0.9727인데 검증은 0.8213으로 간격이 0.1514다. 학습에 쓴 데이터에서만 점수가 높게 나오는 경향이 남아 있다.
hold-out Test는 모든 선택이 끝난 뒤 한 번만 사용하는데 소수점까지 정확하다라고 보기보다 오차를 감안한 추정치로 받아들이는 것이 적절하다.
CV와 Test 모두 같은 데이터를 쪼개 쓴 것이므로, 이 데이터가 특정 지역·시기에 치우쳐 있는지 이번 실험만으로 확인할 수 없다.

---

# 실습 마무리

1. GridSearchCV와 RandomizedSearchCV의 가장 큰 탐색 방식 차이는 무엇인가요?
-> Grid -> 주어진 모든 후보 조합 평가
-> Random -> n_iter만큼 무작위 조합 평가

2. RandomizedSearchCV의 `n_iter`는 무엇을 의미하나요?
-> Random Search가 평가할 조합의 개수, 탐색 예산을 의미

3. Optuna에서 `study`와 `trial`은 각각 무엇인가요?
-> study -> 전체탐색 //
-> trial -> 한 번의 시도

4. 최종 모델을 고를 때 CV 평균 외에 무엇을 함께 봐야 하나요?
-> CV 평균, CV 표준편차, 탐색 학습 비용
(학습비용 큰 차이 없었다..?)

5. hold-out Test는 언제 사용해야 하나요?
-> 모든 탐색과 모델 선택이 끝난 뒤 마지막에 한번 사용한다.

6. 최종 튜닝 모델과 메타데이터를 저장하는 이유는 무엇인가요?
-> 나중에 같은 모델을 재사용하고,
-> 어떤 조건으로 만들어졌는지 추적(Log)하여 재현성과 운영 관리를 높이기 위해 저장한다.
